# MGMT radiomics baseline pipeline (dummy → RF → GA-RF)

This notebook shows how to:
- load the four pre-split CSV files you created,
- get floor performance using dummy classifiers,
- train a plain Random Forest on all radiomics features,
- plug in a GA-selected feature subset to train GA-RF.
- make ROC curve and show result through hyperparameter tuning


In [12]:
import os
import random
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

import matplotlib.pyplot as plt

## 1. Load data
We load `X_train_xgbsel.csv`, `X_test_xgbsel.csv`, `y_train.csv`, and `y_test.csv`.  
Labels are squeezed to 1D so that scikit-learn can use them directly.  
If your labels are strings like `"Methylated"`, you can map them to integers right after loading.


In [13]:
# set seeds for reproducibility
random.seed(42)
np.random.seed(42)

# read feature matrices
X_train = pd.read_csv("../data/X_train_xgbsel.csv")   # training features
X_test  = pd.read_csv("../data/X_test_xgbsel.csv")    # held-out test features

# read target vectors (squeeze → Series, not DataFrame)
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/y_test.csv").squeeze()

# quick sanity check on shapes
print("X_train:", X_train.shape)   # (n_train, d)
print("X_test: ", X_test.shape)    # (n_test, d)
print("y_train:", y_train.shape)   # (n_train,)
print("y_test: ", y_test.shape)    # (n_test,)

# --- GA will select columns from TRAIN ---
New_FS = X_train.copy()               # GA works on training features
y_trn  = y_train.reset_index(drop=True)

print("GA base features (train):", New_FS.shape)
print("GA base labels  (train):", y_trn.shape)

GA_POP_PATH   = "../data/ga_pop.npy"
GA_SCORE_PATH = "../data/ga_scores.npy"

size  = 50
n_feat = New_FS.shape[1]   # number of features GA will see

# 1) Convert pandas DataFrames to NumPy arrays in advance
X_np = New_FS.to_numpy(dtype=np.float32)
y_np = y_trn.to_numpy()

# 2) Precompute the folds from StratifiedKFold (we’ll reuse these in fitness)
from sklearn.model_selection import StratifiedKFold
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = [(train_idx, test_idx) for train_idx, test_idx in kfold.split(X_np, y_np)]


# if labels are text, map to 0/1 here
# y_trn = (y_trn == "Methylated").astype(int)
# y_test = (y_test == "Methylated").astype(int)


X_train: (53, 38)
X_test:  (65, 38)
y_train: (53,)
y_test:  (65,)
GA base features (train): (53, 38)
GA base labels  (train): (53,)


## 2. Dummy baselines
We train two dummy models to see the minimum performance on this split.  
`most_frequent` = always predict the majority class.  
`stratified` = predict classes according to training distribution (a bit harder baseline).


In [14]:
# dummy that always predicts the most frequent class in y_train
dummy_mf = DummyClassifier(strategy="most_frequent")
dummy_mf.fit(X_train, y_train)
y_pred_mf = dummy_mf.predict(X_test)
acc_mf = accuracy_score(y_test, y_pred_mf)
print(f"Dummy (most_frequent) accuracy on TEST: {acc_mf:.3f}")

# dummy that samples labels according to class proportion in y_train
dummy_st = DummyClassifier(strategy="stratified", random_state=42)
dummy_st.fit(X_train, y_train)
y_pred_st = dummy_st.predict(X_test)
acc_st = accuracy_score(y_test, y_pred_st)
print(f"Dummy (stratified) accuracy on TEST:   {acc_st:.3f}")

Dummy (most_frequent) accuracy on TEST: 0.200
Dummy (stratified) accuracy on TEST:   0.462


## 3. Plain Random Forest (no GA)

We now train a real model on **all 38 radiomics features** using 54 training cases.  
Because the training set is small and high-dimensional, we first run a 5-fold **stratified** cross-validation on the training set to get a more stable estimate.  



In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample",
)

# internal CV on the TRAIN set
cv_acc = cross_val_score(rf, X_train, y_train, cv=cv, scoring="accuracy")
print("RF 5-fold CV accuracy:", cv_acc)
print("RF 5-fold CV accuracy (mean):", cv_acc.mean())

if y_train.nunique() == 2:
    cv_auc = cross_val_score(rf, X_train, y_train, cv=cv, scoring="roc_auc")
    print("RF 5-fold CV AUC:", cv_auc)
    print("RF 5-fold CV AUC (mean):", cv_auc.mean())

# fit on all training samples
rf.fit(X_train, y_train)

# evaluate once on the held-out TEST set
y_pred_test = rf.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_test)
print("\nRF TEST accuracy:", round(acc_test, 3))

if y_test.nunique() == 2:
    y_proba_test = rf.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, y_proba_test)
    print("RF TEST AUC:     ", round(auc_test, 3))

print("\nRF TEST classification report:")
print(classification_report(y_test, y_pred_test))

RF 5-fold CV accuracy: [0.72727273 0.63636364 1.         0.9        0.7       ]
RF 5-fold CV accuracy (mean): 0.7927272727272727
RF 5-fold CV AUC: [0.83333333 0.73333333 1.         0.88       0.88      ]
RF 5-fold CV AUC (mean): 0.8653333333333333

RF TEST accuracy: 0.692
RF TEST AUC:      0.627

RF TEST classification report:
              precision    recall  f1-score   support

           0       0.27      0.31      0.29        13
           1       0.82      0.79      0.80        52

    accuracy                           0.69        65
   macro avg       0.54      0.55      0.54        65
weighted avg       0.71      0.69      0.70        65



## 4. GA-RF (paper-faithful version)

Below is a version that matches the 2022 code style much more closely.

Key points:

- we recreate their globals: `New_FS`, `y_trn`, `kfold`, `model`
- we keep their function names: `initilization_of_population`, `fitness_score`, `selection`, `crossover`, `mutation`, `generations`
- we fix only the things that would break in pandas/NumPy now:
  - `np.bool` → `bool`
  - `.iloc[train].iloc[:,chromosome]` → `.iloc[train, chromosome]`
- at the end we **actually call** `generations(...)` so you see `gen 0 ...`, `gen 1 ...` printed like their script
- this version uses **your** data: `X_train` → `New_FS`, `y_train` → `y_trn`

You can change `n_gen` or `size` later. After this, in step 5, you can take `best_chromo[-1]` and train a clean RF/XGB on that subset.


In [ ]:
# -------------------------------
# 0) Reproducibility & data setup
# -------------------------------
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# GA will search over training features (train-only!)
New_FS = X_train.copy()
y_trn  = y_train.reset_index(drop=True)

# Precompute numpy arrays and fixed CV folds (train-only)
X_np = New_FS.to_numpy()
y_np = y_trn.to_numpy()
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_np, y_np))

# Model factory for GA-internal CV (avoid nested parallelism)
def make_rf(seed=RANDOM_STATE):
    return RandomForestClassifier(
        n_estimators=100,
        n_jobs=1,  # single-thread inside GA CV
        random_state=seed,
        bootstrap=True,
    )
# -------------------------------
# 1) GA/global hyperparameters
# -------------------------------
size   = 50
n_feat = New_FS.shape[1]


In [17]:
# -----------------------------------------
# 2) population init + fitness (AUC-based)
# -----------------------------------------
def initilization_of_population(size, n_feat, off_ratio=0.3, rng=None):
    """
    Create an initial list of chromosomes (boolean masks over features).
    About `off_ratio` will be OFF (False); the rest ON (True), then shuffled.
    """
    if rng is None:
        rng = np.random.default_rng(RANDOM_STATE)
    population = []
    off_cnt = int(off_ratio * n_feat)
    for _ in range(size):
        chromosome = np.ones(n_feat, dtype=bool)
        chromosome[:off_cnt] = False
        rng.shuffle(chromosome)
        population.append(chromosome)
    return population


def fitness_score(population, model=None):
    """
    Evaluate each chromosome by mean ROC-AUC over precomputed folds.
    Uses global X_np, y_np, folds.
    `model` is a factory (callable) or a sklearn estimator; we clone per fold.
    Returns scores (desc), sorted population, weights, and aggregate confusion counts.
    """
    global X_np, y_np, folds

    scores, newtp, newfp, newtn, newfn = [], [], [], [], []

    for chromosome in population:
        # boolean mask -> column indices
        col_idx = np.where(chromosome)[0]

        # empty subset guard
        if col_idx.size == 0:
            scores.append(0.0)
            newtp.append(0); newfp.append(0); newtn.append(0); newfn.append(0)
            continue

        tp_list, fp_list, tn_list, fn_list, auc_list = [], [], [], [], []

        # fold loop
        for train_idx, test_idx in folds:
            X_tr = X_np[train_idx][:, col_idx]
            X_te = X_np[test_idx][:, col_idx]
            y_tr = y_np[train_idx]
            y_te = y_np[test_idx]

            # fresh model per fold
            if callable(model):
                clf = model()
            else:
                clf = clone(model)

            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_te)[:, 1]
            auc_i = roc_auc_score(y_te, proba)
            auc_list.append(auc_i)

            # (optional) confusion at 0.5 threshold (not used for fitness)
            preds = (proba >= 0.5).astype(int)
            tn_i, fp_i, fn_i, tp_i = confusion_matrix(y_te, preds, labels=[0, 1]).ravel()
            tp_list.append(tp_i); fp_list.append(fp_i)
            tn_list.append(tn_i); fn_list.append(fn_i)

        # aggregate over folds
        scores.append(float(np.mean(auc_list)))
        newtp.append(int(np.sum(tp_list)))
        newfp.append(int(np.sum(fp_list)))
        newtn.append(int(np.sum(tn_list)))
        newfn.append(int(np.sum(fn_list)))

    # sort by score desc
    scores = np.array(scores, dtype=float)
    population = np.array(population, dtype=object)

    total_score = scores.sum()
    if total_score <= 0:
        weights = np.ones_like(scores) / len(scores)
    else:
        weights = scores / total_score

    newtp = np.array(newtp); newfp = np.array(newfp)
    newtn = np.array(newtn); newfn = np.array(newfn)
    inds = np.argsort(scores)[::-1]

    return (
        list(scores[inds]),
        list(population[inds]),
        list(weights[inds]),
        list(newtp[inds]),
        list(newfp[inds]),
        list(newtn[inds]),
        list(newfn[inds]),
    )

In [18]:

# ---------------------------------------------
# 3) GA operators (selection / crossover / mut)
# ---------------------------------------------
def selection(pop_after_fit, weights, k):
    """
    Roulette-wheel selection: sample k chromosomes according to fitness weights.
    """
    return list(random.choices(pop_after_fit, weights=weights, k=k))


def crossover(p1, p2, crossover_rate):
    """
    Single-point crossover. Two parents -> two children.
    With prob `crossover_rate`, swap tails after a random point; else copy.
    """
    c1, c2 = p1.copy(), p2.copy()
    if random.random() < crossover_rate and len(p1) > 2:
        pt = random.randint(1, len(p1) - 2)
        c1 = np.concatenate((p1[:pt], p2[pt:]))
        c2 = np.concatenate((p2[:pt], p1[pt:]))
    return [c1, c2]


def mutation(chromosome, mutation_rate):
    """
    Bit-flip mutation with probability = mutation_rate per gene.
    Returns a NEW chromosome.
    """
    mutated = chromosome.copy()
    for i in range(len(mutated)):
        if random.random() < mutation_rate:
            mutated[i] = not mutated[i]
    return mutated

In [19]:

# ---------------------------------------
# 4) Full GA loop (original multi-gen)
# ---------------------------------------
def generations(size, n_feat, crossover_rate, mutation_rate, n_gen, model):
    """
    Run GA for n_gen generations (train-only CV) and return histories:
      best_chromo_per_gen, best_score_per_gen
    """
    best_chromo = []
    best_score = []

    # start with random population
    population_nextgen = initilization_of_population(size, n_feat)

    for gen in range(n_gen):
        scores, pop_after_fit, weights, tp, fp, tn, fn = fitness_score(population_nextgen, model=model)

        # best of this generation
        top_score = scores[0]
        top_chrom = pop_after_fit[0]
        print(f"[gen {gen}] best_auc={top_score:.4f} | n_features={int(top_chrom.sum())}")

        # elitism (keep top-2)
        elites = pop_after_fit[:2]
        k = size - 2

        # select parents
        parents = selection(pop_after_fit, weights, k)

        # crossover + mutation
        children = []
        for i in range(0, len(parents), 2):
            p1 = parents[i]
            p2 = parents[(i + 1) % len(parents)]
            for child in crossover(p1, p2, crossover_rate):
                child = mutation(child, mutation_rate)   # <-- APPLY mutation result
                children.append(child)

        # next population (elites first)
        population_nextgen = []
        population_nextgen.extend(elites)
        for c in children:
            if len(population_nextgen) < size:
                population_nextgen.append(c)

        # history
        best_chromo.append(top_chrom)
        best_score.append(top_score)

    return best_chromo, best_score

In [20]:

# ---------------------------------------
# 5) Single GA step (one generation)
# ---------------------------------------
def ga_one_step(population, size, crossover_rate, mutation_rate, model):
    """
    Run exactly 1 generation; return:
      next_population, best_chromo_of_gen, best_score_of_gen
    """
    scores, pop_after_fit, weights, tp, fp, tn, fn = fitness_score(population, model=model)

    # best in this generation
    best_score  = scores[0]
    best_chromo = pop_after_fit[0]

    # elitism (top-2)
    elites = pop_after_fit[:2]
    k = size - 2

    # select parents
    parents = selection(pop_after_fit, weights, k)

    # children
    children = []
    for i in range(0, len(parents), 2):
        p1 = parents[i]
        p2 = parents[(i + 1) % len(parents)]  # wrap around
        for child in crossover(p1, p2, crossover_rate):
            child = mutation(child, mutation_rate)       # <-- APPLY mutation result
            children.append(child)

    # build next population
    next_pop = []
    next_pop.extend(elites)
    for c in children:
        if len(next_pop) < size:
            next_pop.append(c)

    return next_pop, best_chromo, best_score

In [21]:

# ---------------------------------------
# 6) Track top-K solutions across runs
# ---------------------------------------
BEST_PATH = "../data/ga_best.npz"
TOP_K = 20

def update_top_solutions_by_run(best_path, run_id, new_score, new_mask, new_cols, top_k=20):
    """
    Keep at most `top_k` best runs (one entry per run_id).
    If file exists but has old / wrong keys, we re-initialize.
    """
    prev_run_ids, prev_scores, prev_masks, prev_cols = [], [], [], []

    if os.path.exists(best_path):
        try:
            prev = np.load(best_path, allow_pickle=True)
            prev_run_ids = prev["run_ids"].tolist()
            prev_scores  = prev["top_scores"].tolist()
            prev_masks   = prev["top_masks"].tolist()
            prev_cols    = prev["top_cols"].tolist()
        except KeyError:
            prev_run_ids, prev_scores, prev_masks, prev_cols = [], [], [], []

    # update or append
    if run_id in prev_run_ids:
        idx = prev_run_ids.index(run_id)
        if new_score > prev_scores[idx]:
            prev_scores[idx] = float(new_score)
            prev_masks[idx]  = new_mask
            prev_cols[idx]   = new_cols
    else:
        prev_run_ids.append(run_id)
        prev_scores.append(float(new_score))
        prev_masks.append(new_mask)
        prev_cols.append(new_cols)

    # keep top_k by score
    order = np.argsort(prev_scores)[::-1][:top_k]
    top_run_ids = [prev_run_ids[i] for i in order]
    top_scores  = [prev_scores[i]  for i in order]
    top_masks   = [prev_masks[i]   for i in order]
    top_cols    = [prev_cols[i]    for i in order]

    # save back
    np.savez(
        best_path,
        run_ids=np.array(top_run_ids, dtype=int),
        top_scores=np.array(top_scores, dtype=float),
        top_masks=np.array(top_masks, dtype=object),
        top_cols=np.array(top_cols, dtype=object),
    )
    return top_run_ids, top_scores


## Repeated GA with stratified 5-fold CV (20×) 

In [22]:
N_REPEATS = 5
GENS_PER_RUN = 1

for run in range(N_REPEATS):
    # 1) No need to create a new kfold here anymore
    # kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + run)

    # 2) initialize population
    pop = initilization_of_population(size=size, n_feat=n_feat)
    history_chromo, history_score = [], []

    for gen in range(GENS_PER_RUN):
        pop, bc, bs = ga_one_step(
            pop,
            size=size,
            crossover_rate=0.8,
            mutation_rate=0.05,
            model= make_rf,       # keep this (preferably pass a factory)
        )
        history_chromo.append(bc)
        history_score.append(bs)

    # pick the best solution inside THIS run
    best_idx   = int(np.argmax(history_score))
    best_score = float(history_score[best_idx])
    best_mask  = history_chromo[best_idx]
    best_cols  = np.where(best_mask)[0]

    run_ids, top_scores = update_top_solutions_by_run(
        BEST_PATH,
        run_id=run,
        new_score=best_score,
        new_mask=best_mask,
        new_cols=best_cols,
        top_k=5,
    )

    print("top runs:", run_ids)
    print("top scores:", top_scores)


top runs: [14, 7, 3, 2, 9]
top scores: [0.9254545454545454, 0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272]
top runs: [7, 14, 3, 9, 2]
top scores: [0.9254545454545454, 0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272]
top runs: [7, 14, 3, 2, 9]
top scores: [0.9254545454545454, 0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272]
top runs: [7, 14, 3, 9, 2]
top scores: [0.9254545454545454, 0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272]
top runs: [14, 7, 3, 2, 9]
top scores: [0.9254545454545454, 0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272]


# Show the result 

In [23]:
best = np.load("../data/ga_best.npz", allow_pickle=True)
run_ids    = best["run_ids"].tolist()
top_scores = best["top_scores"].tolist()
top_cols   = best["top_cols"].tolist()

print("=== TOP RUNS ===")
for rid, s, cols in zip(run_ids, top_scores, top_cols):
    print(f"run {rid}: score={s:.4f}, cols(first 5)={cols[:5]}, total={len(cols)}")


=== TOP RUNS ===
run 14: score=0.9255, cols(first 5)=[0 1 2 3 4], total=20
run 7: score=0.9255, cols(first 5)=[0 1 2 3 4], total=20
run 3: score=0.9236, cols(first 5)=[0 1 3 5 6], total=30
run 2: score=0.9073, cols(first 5)=[1 2 4 5 6], total=26
run 9: score=0.9073, cols(first 5)=[0 1 2 3 4], total=29
